# Parallel Table Extraction

This notebook demonstrates how to use parallel processing to speed up table extraction from PDF documents.

For large PDF documents with many pages, table extraction can be time-consuming. By using multiprocessing, you can extract tables from multiple pages simultaneously, significantly reducing processing time.

**Note:** ChunkNorris automatically uses multiprocessing for documents with 20+ pages (configurable). For smaller documents, it uses sequential processing to avoid multiprocessing overhead.

## Installation

Make sure you have ChunkNorris installed:

```bash
pip install chunknorris
```

## Basic Example: Auto vs Manual Configuration

Let's compare automatic multiprocessing detection with manual configuration.

In [ ]:
import os
from multiprocessing import cpu_count
from time import perf_counter
from chunknorris.parsers import PdfParser

# Display available CPU cores
print(f"Available CPU cores: {cpu_count()}")

### Auto-detection (Default)

By default, ChunkNorris automatically decides whether to use multiprocessing based on document size:
- Documents with **20+ pages**: Uses multiprocessing with `cpu_count()` workers
- Documents with **< 20 pages**: Uses sequential processing

In [ ]:
# Replace with your PDF file path
pdf_filepath = "example_data/dummy.pdf"

# Auto-detection (default behavior)
parser_auto = PdfParser(extract_tables=True)

start = perf_counter()
doc_auto = parser_auto.parse_file(pdf_filepath)
time_auto = perf_counter() - start

print(f"Auto-detection (default):")
print(f"  Tables found: {len(parser_auto.tables)}")
print(f"  Time taken: {time_auto:.2f} seconds")

### Force Sequential Processing

You can force sequential processing even for large documents by setting `table_extraction_max_workers=1`:

In [ ]:
# Force sequential processing
parser_seq = PdfParser(
    extract_tables=True,
    table_extraction_max_workers=1  # Force sequential processing
)

start = perf_counter()
doc_seq = parser_seq.parse_file(pdf_filepath)
time_seq = perf_counter() - start

print(f"Sequential processing (forced):")
print(f"  Tables found: {len(parser_seq.tables)}")
print(f"  Time taken: {time_seq:.2f} seconds")

## Customizing Multiprocessing Behavior

You have full control over the multiprocessing configuration:

In [ ]:
# Example 1: Use a specific number of workers
parser_custom = PdfParser(
    extract_tables=True,
    table_extraction_max_workers=4  # Use exactly 4 worker processes
)

start = perf_counter()
doc_custom = parser_custom.parse_file(pdf_filepath)
time_custom = perf_counter() - start

print(f"Custom workers (workers=4):")
print(f"  Tables found: {len(parser_custom.tables)}")
print(f"  Time taken: {time_custom:.2f} seconds")

# Example 2: Customize the page threshold for auto-detection
parser_custom_threshold = PdfParser(
    extract_tables=True,
    table_extraction_multiproc_page_thresh=10  # Auto-enable multiprocessing for 10+ pages instead of 20+
)

start = perf_counter()
doc_custom_threshold = parser_custom_threshold.parse_file(pdf_filepath)
time_custom_threshold = perf_counter() - start

print(f"\nCustom threshold (threshold=10 pages):")
print(f"  Tables found: {len(parser_custom_threshold.tables)}")
print(f"  Time taken: {time_custom_threshold:.2f} seconds")

## Performance Tips

### Auto-detection vs Manual Configuration:

- ✅ **Use auto-detection (default)**: Best for most use cases - automatically optimizes based on document size
- ✅ **Customize threshold**: If you process many documents of similar size, adjust `table_extraction_multiproc_page_thresh`
- ✅ **Force workers**: Use `table_extraction_max_workers` when you need explicit control

### When multiprocessing provides the most benefit:

- ✅ **Large documents** (20+ pages by default): Parallel processing can provide 2-4x speedup
- ✅ **Documents with many tables**: More tables = more benefit from parallelization
- ✅ **Batch processing**: Processing multiple documents in succession

### When to use sequential processing:

- ❌ **Small documents** (< 20 pages by default): Multiprocessing overhead may be slower
- ❌ **Few or no tables**: No tables = no benefit from parallelization
- ❌ **Memory-constrained systems**: Each worker opens the document in memory

### Choosing the number of workers:

- **Default (`None`)**: Automatically uses `cpu_count()` for large documents, sequential for small ones
- **`cpu_count()`**: Use all available CPU cores (good for large documents)
- **`cpu_count() // 2`**: On systems with many CPUs, avoid overloading
- **Custom value**: Experiment to find optimal configuration for your use case
- **`1`**: Force sequential processing

## Complete Example

Here's a complete example that processes a PDF and displays the extracted tables:

In [ ]:
from chunknorris.parsers import PdfParser
from multiprocessing import cpu_count

# Parse PDF with parallel table extraction
parser = PdfParser(
    extract_tables=True,
    table_extraction_max_workers=cpu_count()
)

doc = parser.parse_file("example_data/dummy.pdf")

# Display extracted tables
print(f"Found {len(parser.tables)} table(s):\n")

for i, table in enumerate(parser.tables, 1):
    print(f"Table {i} (Page {table.page}):")
    print(table.to_markdown())
    print("\n" + "="*60 + "\n")

## Technical Details

### How it works:

1. The PDF document is divided by pages
2. Each worker process:
   - Opens the PDF document
   - Extracts tables from assigned pages
   - Returns the results to the main process
3. Results are collected (using `as_completed` for better parallelism) and sorted by page order

### Auto-detection logic:

When `table_extraction_max_workers=None` (default):
- If `page_count >= table_extraction_multiproc_page_thresh`: Uses `cpu_count()` workers
- Otherwise: Uses sequential processing (1 worker)

### Configuration parameters:

- **`table_extraction_max_workers`** (int | None):
  - `None` (default): Auto-detect based on document size
  - `1`: Force sequential processing
  - `> 1`: Use specified number of workers

- **`table_extraction_multiproc_page_thresh`** (int):
  - Default: `20`
  - Minimum pages required to auto-enable multiprocessing
  - Only applies when `table_extraction_max_workers=None`

### Implementation notes:

- Uses Python's `ProcessPoolExecutor` for multiprocessing
- Uses `as_completed()` instead of `map()` for better parallelism (doesn't block on slow pages)
- Each worker reopens the PDF (required by PyMuPDF's thread-safety limitations)
- Workers include error handling to prevent single page failures from crashing entire extraction
- Works with both file paths and byte streams